In [3]:
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from PyPDF2 import PdfReader
import re
import os

# Load environment variables
load_dotenv()

True

In [4]:
model = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("BASE_URL"),
    openai_api_key=os.getenv("OPENAI_API_KEY")
)


In [5]:
from crewai.tools import tool

@tool("fetch_pdf_content")
def fetch_pdf_content(pdf_path: str) -> str:
    """Reads a local PDF and returns the content.

    Args:
        pdf_path: Path to the PDF file to read

    Returns:
        Extracted text content from the PDF
    """
    try:
        with open(pdf_path, 'rb') as f:
            pdf = PdfReader(f)
            text = '\n'.join(page.extract_text() for page in pdf.pages if page.extract_text())

        processed_text = re.sub(r'\s+', ' ', text).strip()
        return processed_text
    except Exception as e:
        return f"Error reading PDF: {str(e)}"


In [6]:
pdf_reader = Agent(
    role='PDF Content Extractor',
    goal='Extract and preprocess text from a PDF located in current local directory',
    backstory='Specializes in handling and interpreting PDF documents',
    verbose=True,
    tools=[fetch_pdf_content],
    allow_delegation=False,
    llm=model
)

article_writer = Agent(
    role='Article Creator',
    goal='Write a concise and engaging article',
    backstory='Expert in creating informative and engaging articles',
    verbose=True,
    allow_delegation=False,
    llm=model
)

title_creator = Agent(
    role='Title Generator',
    goal='Generate a compelling title for the article',
    backstory='Skilled in crafting engaging and relevant titles',
    verbose=True,
    allow_delegation=False,
    llm=model
)

In [7]:
def pdf_reading_task(pdf_path):
    return Task(
        description=f"Read the PDF file at {pdf_path} and extract all text content. Process the text by removing extra whitespace and return the cleaned content.",
        agent=pdf_reader,
        expected_output="Extracted and preprocessed text from the PDF",
    )


task_article_drafting = Task(
    description="Create a concise article with 8-10 paragraphs based on the extracted PDF content.",
    agent=article_writer,
    expected_output="8-10 paragraphs describing the key points of the PDF",
)

task_title_generation = Task(
    description="Generate an engaging and relevant title for the article.",
    agent=title_creator,
    expected_output="A Title of About 5-7 Words"
)

In [8]:
# Define the PDF path from environment variable
pdf_local_relative_path = os.getenv("PDF_PATH")

crew = Crew(
    agents=[pdf_reader, article_writer, title_creator],
    tasks=[pdf_reading_task(pdf_local_relative_path),
           task_article_drafting,
           task_title_generation],
    verbose=True  # Changed from verbose=2 to verbose=True
)

# Let's start!
result = crew.kickoff()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d55eb876-b333-4a1a-86b2-8ffe34fe100c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: PDF Content Extractor                                                                                   │
│                                                                                                                 │
│  Task: Read the PDF file at ebook-springboot-4edicao.pdf and extract all text content. Process the text by      │
│  removing extra whitespace and return the cleaned content.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

C:\Users\jv_rs\PycharmProjects\AIAgent\.venv\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\jv_rs\PycharmProjects\AIAgent\.venv\Lib\site-packages\rich\live.py:256: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: PDF Content Extractor                                                                                   │
│                                                                                                                 │
│  Thought: Action: fetch_pdf_content                                                                             │
│                                                                                                                 │
│  Using Tool: fetch_pdf_content                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Title Generator                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Spring Boot REST API and Microservices: Mastering Spring Cloud                                                 │
│                                                                                                                 │
│  This book covers the development of REST APIs and microservices using Spring Boot. It presents basic concepts  │
│  of Spring, such as MVC, and explains how to create a simple REST API. Then, it discusses project organization  │
│  and Maven, unit testing, API documentation, JWT-based security, authentication, and authorization, version     │
│  control, and continuous integration. The book also covers creating microservices with Spring Cloud, including  │
│  service discovery, configurations, and gateways, as well as explaining how to use Spring Data for database     │
│  communication. The author provides practical examples throughout the book, using Java and Spring Boot to       │
│  illustrate theoretical concepts.                                                                               │
│                                                                                                                 │
│  The book includes complete applications available on GitHub, making it easier for readers to test ideas in     │
│  their own development environments. It also features a video supplement covering the main Spring Cloud         │
│  projects applied to microservices architecture. The book is well-organized and written clearly, presenting     │
│  complex concepts in an easy-to-understand manner and providing examples throughout the way. The author does    │
│  an excellent job of presenting the components of the Spring platform and their relationships, making           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 4c2f8ba5-17a6-4733-bafd-b23996a15e0d                                                                     │
│  Agent: Title Generator                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d55eb876-b333-4a1a-86b2-8ffe34fe100c                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Spring Boot REST API and Microservices: Mastering Spring Cloud                                   │
│                                                                                                                 │
│  This book covers the development of REST APIs and microservices using Spring Boot. It presents basic concepts  │
│  of Spring, such as MVC, and explains how to create a simple REST API. Then, it discusses project organization  │
│  and Maven, unit testing, API documentation, JWT-based security, authentication, and authorization, version     │
│  control, and continuous integration. The book also covers creating microservices with Spring Cloud, including  │
│  service discovery, configurations, and gateways, as well as explaining how to use Spring Data for database     │
│  communication. The author provides practical examples throughout the book, using Java and Spring Boot to       │
│  illustrate theoretical concepts.                                                                               │
│                                                                                                                 │
│  The book includes complete applications available on GitHub, making it easier for readers to test ideas in     │
│  their own development environments. It also features a video supplement covering the main Spring Cloud         │
│  projects applied to microservices architecture. The book is well-organized and written clearly, presenting     │
│  complex concepts in an easy-to-understand manner and providing examples throughout the way. The author does    │
│  an excellent job of presenting the components of the Spring platform and their relationships, making           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 